# Clash Royale BC Pre-training (Colab)

Behavioral Cloning from Observation (BCO): train an Inverse Dynamics Model on
live rollouts (which have action labels), use it to label action-free
demonstration pairs extracted from YouTube videos, then clone the labeled
demonstrations into a PPO policy. Download the result and use it as the
starting checkpoint for live collection.

Inputs you upload: one or more `rollouts/*.npz` from the Mac, plus either
gameplay `.mp4` files or a pre-extracted `pairs.npz` (made on the Mac with
`scripts/extract_demonstrations.py`).

In [ ]:
!git clone https://github.com/alex-h-sun/RL_gaming_agent.git repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip install -q -r requirements-train.txt yt-dlp

In [ ]:
# Upload live rollout file(s) for IDM training
from google.colab import files
import os
os.makedirs('rollouts', exist_ok=True)
os.makedirs('data/demos', exist_ok=True)
uploaded = files.upload()
rollout_paths = []
for name in uploaded:
    target = f'rollouts/{name}'
    os.replace(name, target)
    rollout_paths.append(target)
print(rollout_paths)

In [ ]:
# Train the IDM (obs_t, obs_t+1 -> action) on the rollouts
!python -m scripts.train_idm {' '.join(rollout_paths)} --out checkpoints/idm.pt --epochs 10

## Demonstration pairs

Option A (recommended): download videos and extract here on Colab.
Set `VIDEO_URLS`, then preview the crop on one frame before extracting —
adjust `CROP` (left,top,right,bottom fractions trimmed) until only the
gameplay arena is visible (no pillarboxing, no facecam, no overlays).

Option B: skip the next two cells and upload a `pairs.npz` you extracted
locally; place it at `data/demos/pairs.npz`.

In [ ]:
VIDEO_URLS = [
    # TODO: paste YouTube URLs of high-level Clash Royale ladder gameplay
]
CROP = '0.0,0.0,0.0,0.0'
assert VIDEO_URLS, 'Add at least one video URL above'
!python -m scripts.download_videos {' '.join(VIDEO_URLS)} --out data/youtube

In [ ]:
# Preview the crop, then extract pairs once it looks right
!python -m scripts.extract_demonstrations data/youtube/*.mp4 --crop {CROP} --preview-frame
from IPython.display import Image, display
import glob
for png in glob.glob('*_crop_preview.png'):
    display(Image(png, width=300))

In [ ]:
!python -m scripts.extract_demonstrations data/youtube/*.mp4 --crop {CROP} --fps 4 --out data/demos/pairs.npz

In [ ]:
# Label pairs with the IDM and run behavioral cloning
!python -m scripts.train_bc data/demos/pairs.npz --idm checkpoints/idm.pt --out checkpoints/bc_pretrained.zip --epochs 5

In [ ]:
# Download the pretrained checkpoint; on the Mac, copy it to checkpoints/best.zip
from google.colab import files
files.download('checkpoints/bc_pretrained.zip')